# Differences between round and elongated osteocytes

The groups come from the tags given by eye (`annotate.py`). With `GROUP_SOURCE = "tags+predicted"`, cells without a
round / elongated tag (the real-but-dirty ones) are added with the group predicted by `phenotype_classifier.ipynb`,
when its probability is at least `MIN_PROBABILITY`.

Every feature is compared between the groups: medians, Cliff's δ (rank effect size, −1 to 1; |δ| ≈ 0.15 small,
0.33 medium, 0.47 large; Romano et al. 2006) and a p-value from shuffling the group labels **within each stack**, so
differences between stacks cannot create an effect; Holm correction over all features. Then the location of the
groups relative to the main axis of each stack's hull, maps, and example cells.

## Settings

In [ ]:
from pathlib import Path

GROUP_SOURCE = "tags"            # "tags" or "tags+predicted"
MIN_PROBABILITY = 0.8            # for predicted groups
COMPARE = ["axis_ratio_2_1", "body_elongation", "body_flatness", "body_volume_um3", "body_axis_1_um", "process_fraction",
           "n_dendrites", "mean_dendrite_reach_um", "max_dendrite_reach_um", "total_dendrite_path_um",
           "dendrite_starriness", "dendrite_planarity", "dendrite_randomness", "dendrite_axis_alignment",
           "nucleus_volume_um3", "radial_position", "distance_to_axis_um"]
LOG_SCALE = {"body_volume_um3", "total_dendrite_path_um", "nucleus_volume_um3"}
N_PERM = 10000
N_EXAMPLES = 10                  # random example cells per group (0 = none)
EXAMPLE_FILE = "_rotation.gif"   # _rotation.gif, _mip.png, _dendrites.gif or _dendrites.png
RESULTS = Path("results")
SEED = 20261014

In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from statsmodels.stats.multitest import multipletests

from osteo_common import (DATA, FEATURE_LABELS, GROUP_COLOURS, INK, add_features, add_positions, clean, cliffs_delta,
                          load, series_tag, style, within_series_permutation)

style()
RESULTS.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

## Data and groups

In [ ]:
all_cells, dendrites = load()
cells, summary = clean(all_cells)
cells = add_features(cells, dendrites)
cells, geometry = add_positions(cells, all_cells, dendrites, rng)
cells["group"] = cells["phenotype"].replace("", np.nan)
cells["group_source"] = np.where(cells["group"].notna(), "tag", "")
if GROUP_SOURCE == "tags+predicted":
    if not (RESULTS / "predicted_phenotypes.csv").exists():
        raise FileNotFoundError(f"{RESULTS / 'predicted_phenotypes.csv'} not found: run phenotype_classifier.ipynb first "
                                "(from this folder), or set GROUP_SOURCE = 'tags'")
    pred = pd.read_csv(RESULTS / "predicted_phenotypes.csv")[["series", "label", "p_round"]]
    cells = cells.merge(pred, on=["series", "label"], how="left")
    sure = cells["group"].isna() & (np.maximum(cells["p_round"], 1 - cells["p_round"]) >= MIN_PROBABILITY)
    cells.loc[sure, "group"] = np.where(cells.loc[sure, "p_round"] >= 0.5, "round", "elongated")
    cells.loc[sure, "group_source"] = "predicted"
grouped = cells.dropna(subset=["group"])
print(f"{len(cells)} kept cells, {len(grouped)} in a group "
      f"({(grouped['group_source'] == 'tag').sum()} by tag, {(grouped['group_source'] == 'predicted').sum()} predicted)")
pd.crosstab(grouped["series"], grouped["group"], margins=True)

## Feature comparison

In [ ]:
rows = []
for f in COMPARE:
    d = grouped.dropna(subset=[f])
    v = np.log10(d[f]) if f in LOG_SCALE else d[f]
    is_round = (d["group"] == "round").to_numpy()
    diff, p = within_series_permutation(v.to_numpy(), is_round, d["series"].to_numpy(), rng, N_PERM)
    a, b = d.loc[~is_round, f], d.loc[is_round, f]
    rows.append({"feature": FEATURE_LABELS.get(f, f), "n": len(d),
                 "elongated": f"{a.median():.3g} [{a.quantile(.25):.3g}, {a.quantile(.75):.3g}]",
                 "round": f"{b.median():.3g} [{b.quantile(.25):.3g}, {b.quantile(.75):.3g}]",
                 "Cliff's δ (round vs elongated)": cliffs_delta(a, b), "p (within-stack)": p})
comparison = pd.DataFrame(rows)
comparison["p (Holm)"] = multipletests(comparison["p (within-stack)"], method="holm")[1]
comparison.to_csv(RESULTS / "group_differences.csv", index=False)
comparison.set_index("feature").round(3)

In [ ]:
ncols = 5
nrows = int(np.ceil(len(COMPARE) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), squeeze=False)
for ax, f in zip(axes.flat, COMPARE):
    for k, (name, colour) in enumerate(GROUP_COLOURS.items()):
        v = grouped.loc[grouped["group"] == name, f].dropna()
        x = k + rng.uniform(-0.18, 0.18, len(v))
        ax.scatter(x, v, s=10, color=colour, alpha=0.6, edgecolor="none")
        q1, med, q3 = v.quantile([.25, .5, .75])
        ax.plot([k - 0.28, k + 0.28], [med, med], color=INK["primary"], linewidth=2)
        ax.vlines(k, q1, q3, color=INK["primary"], linewidth=1)
    ax.set(xticks=[0, 1], xticklabels=list(GROUP_COLOURS), yscale="log" if f in LOG_SCALE else "linear")
    p = comparison.loc[comparison["feature"] == FEATURE_LABELS.get(f, f), "p (Holm)"].iloc[0]
    ax.set_title(f"{FEATURE_LABELS.get(f, f)}\nHolm p = {p:.2g}", fontsize=8)
for ax in axes.flat[len(COMPARE):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS / "group_differences.png")

## Location relative to the hull axis

Per stack, in the image plane: maps of the two groups over the hull of all cells (centroids and dendrite tips) and its main axis.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for name, colour in GROUP_COLOURS.items():
    v = np.sort(grouped.loc[grouped["group"] == name, "radial_position"].dropna().to_numpy())
    ax.step(v, np.arange(1, len(v) + 1) / len(v), color=colour, linewidth=2, label=f"{name} (n = {len(v)})")
ax.set(xlabel="relative radial position (0 = axis, 1 = hull)", ylabel="cumulative fraction", xlim=(0, 1))
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(RESULTS / "group_location.png")

n = len(geometry)
fig, axes = plt.subplots(int(np.ceil(n / 4)), 4, figsize=(16, 4 * np.ceil(n / 4)), squeeze=False)
for ax, (s, (centre, axis, hull, pts)) in zip(axes.flat, geometry.items()):
    ax.fill(pts[hull.vertices, 1], pts[hull.vertices, 0], color=INK["grid"], zorder=0)
    span = np.linalg.norm(pts.max(0) - pts.min(0)) / 2
    ends = centre + np.outer([-span, span], axis)
    ax.plot(ends[:, 1], ends[:, 0], color=INK["primary"], linewidth=1.5, linestyle="--")
    g = cells[cells["series"] == s]
    ax.scatter(g["centroid_x_um"], g["centroid_y_um"], c=g["group"].map(GROUP_COLOURS).fillna(INK["context"]),
               s=30, edgecolor="white", linewidth=0.8, zorder=3)
    ax.set(title=s, aspect="equal", xlabel="x (µm)", ylabel="y (µm)")
    ax.invert_yaxis()
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS / "group_maps.png")

## Examples

`N_EXAMPLES` random cells per group: one rendering each (`EXAMPLE_FILE`) in `results/group_examples/<group>/`, plus a contact sheet of their MIPs.

In [ ]:
EXAMPLES = RESULTS / "group_examples"
shutil.rmtree(EXAMPLES, ignore_errors=True)
for name in GROUP_COLOURS:
    members = grouped[grouped["group"] == name]
    pick = members.sample(min(N_EXAMPLES, len(members)), random_state=SEED) if N_EXAMPLES else members.iloc[:0]
    if not len(pick):
        continue
    (EXAMPLES / name).mkdir(parents=True)
    fig, axes = plt.subplots(int(np.ceil(len(pick) / 5)), 5, figsize=(16, 3.6 * np.ceil(len(pick) / 5)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, pick.iterrows()):
        cell_dir = DATA.parent / series_tag(r["series"]) / "cells"
        f = cell_dir / f"osteocyte_{r['label']}{EXAMPLE_FILE}"
        if f.exists():
            shutil.copy2(f, EXAMPLES / name / f"{series_tag(r['series'])}_{r['label']}{EXAMPLE_FILE}")
        mip = cell_dir / f"osteocyte_{r['label']}_mip.png"
        if mip.exists():
            ax.imshow(Image.open(mip))
        ax.set_title(f"{r['series'].replace('_ultrasonic cleaned', '')} #{r['label']} ({r['group_source']})", fontsize=8)
        ax.set_axis_off()
    for ax in axes.flat[len(pick):]:
        ax.set_visible(False)
    fig.suptitle(name, x=0.01, ha="left", color=GROUP_COLOURS[name])
    fig.tight_layout()
    fig.savefig(EXAMPLES / f"{name}_contact_sheet.png")
    plt.close(fig)
print(f"examples in {EXAMPLES}")

## Save

In [ ]:
cols = ["series", "label", "group", "group_source", *[f for f in COMPARE if f in cells]]
cells[cols].to_csv(RESULTS / "group_cells.csv", index=False)
print(f"wrote {RESULTS / 'group_cells.csv'} ({len(cells)} cells)")